## **Laboratorio: Text-to-SQL con Vanna.ai**

**Text-to-SQL** es la tarea de traducir una pregunta en lenguaje natural a una consulta SQL válida. Hasta hace pocos años esto era un problema de investigación difícil; hoy, con los modelos de lenguaje grandes (LLMs), es una capacidad práctica que permite a usuarios no técnicos interrogar bases de datos con frases como:

> *"¿Cuál es la masa media de los pingüinos macho de la isla Biscoe?"*

y obtener automáticamente el SQL correcto y el resultado.

**Vanna.ai** es una librería open-source de Python que combina dos tecnologías para hacer Text-to-SQL confiable:

1. **RAG (Retrieval-Augmented Generation):** Antes de generar SQL, Vanna recupera del vector store los ejemplos de SQL, definiciones de tablas y documentación más relevantes para la pregunta del usuario. Esto se inyecta en el prompt del LLM como contexto, reduciendo drásticamente las alucinaciones.
2. **LLM:** Con ese contexto enriquecido, el modelo de lenguaje genera la consulta SQL final.

En este laboratorio construirás un sistema Text-to-SQL completo: crearás una base de datos SQLite con los datos de pingüinos del laboratorio anterior, entrenaras a Vanna con el esquema y ejemplos, realizarás consultas en lenguaje natural y, por último, crearás un **chatbot de SQL con Streamlit** que conecta todo lo aprendido.

## Arquitectura del sistema

```
┌──────────────────────────────────────────────────────────┐
│                    Pregunta del usuario                   │
│          "¿Cuántos pingüinos hay en cada isla?"           │
└───────────────────────────┬──────────────────────────────┘
                            │
                            ▼
┌──────────────────────────────────────────────────────────┐
│                   VANNA.AI (RAG + LLM)                   │
│                                                          │
│  1. Busca en ChromaDB los DDL, docs y SQL ejemplos       │
│     más relevantes para la pregunta                      │
│                                                          │
│  2. Construye un prompt enriquecido con ese contexto     │
│                                                          │
│  3. El LLM (GPT-4o-mini) genera el SQL                   │
└───────────────────────────┬──────────────────────────────┘
                            │
                    SQL generado
                            │
                            ▼
┌──────────────────────────────────────────────────────────┐
│               Base de Datos SQLite (local)               │
│                SELECT isla, COUNT(*) ...                  │
└───────────────────────────┬──────────────────────────────┘
                            │
                    DataFrame resultado
                            │
                            ▼
┌──────────────────────────────────────────────────────────┐
│              Interfaz Streamlit (chatbot)                 │
└──────────────────────────────────────────────────────────┘
```

## Objetivos del Laboratorio

1. **Crear una base de datos SQLite** a partir de un DataFrame de pandas.
2. **Configurar Vanna** con ChromaDB (vector store local) y OpenAI como LLM.
3. **Entrenar a Vanna** con DDL, documentación en español y ejemplos SQL.
4. **Realizar consultas en lenguaje natural** y verificar los resultados.
5. **Construir un chatbot de SQL con Streamlit** que integra todo lo anterior.

**Prerequisito:** Tener una clave de API de OpenAI. Si no tienes una, puedes crearla en https://platform.openai.com/api-keys. El modelo `gpt-4o-mini` tiene un coste muy bajo (menos de $0.01 por este laboratorio completo).

**Conexión con el laboratorio anterior:** Usaremos el mismo dataset Palmer Penguins del Lab de Streamlit, pero esta vez cargado en una base de datos SQL real.

**Duración estimada:** 2 horas

---

## Sección 1: Preparación del Entorno

**Celda 1: Instalación de Dependencias**

In [18]:
!pip install --upgrade --quiet "vanna[chromadb,openai]==0.7.9" seaborn pandas plotly streamlit nbformat



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Explicación:**

Vanna se instala con extras opcionales entre corchetes:
- `chromadb` — vector store local para almacenar y recuperar el contexto de entrenamiento
- `openai` — conector al API de OpenAI para usar GPT-4o-mini como LLM generador de SQL

Vanna soporta muchas otras combinaciones: puedes cambiar el LLM por Mistral, Gemini, Ollama (local y gratuito) o el propio servicio cloud de Vanna; y el vector store por Pinecone, Qdrant o pgvector. La arquitectura es modular.

**Celda 2: Imports y Configuración de la API Key**

In [19]:
import seaborn as sns
import pandas as pd
import sqlite3
import os

from vanna.openai import OpenAI_Chat
from vanna.chromadb import ChromaDB_VectorStore

OPENAI_API_KEY = "sk-..."  # ← Reemplaza con tu clave real


**Explicación:**

Simplemente reemplaza `"sk-..."` con tu clave de OpenAI en la línea indicada y ejecuta la celda. La clave solo existe en la memoria del kernel de Jupyter — no se guarda en el archivo del notebook.

> ⚠️ Si vas a compartir el notebook, borra la clave antes de exportarlo o usa `"sk-..."` de nuevo como placeholder.

---
## Sección 2: Crear la Base de Datos SQLite

SQLite es una base de datos relacional embebida en un único archivo. No requiere servidor, es perfecta para prototipos y enseñanza, y Python incluye soporte nativo con el módulo `sqlite3`. Vamos a cargar el dataset Palmer Penguins en una base de datos SQLite para poder interrogarla con SQL.

**Celda 3: Cargar el Dataset y Crear la Base de Datos**

In [20]:
# Cargar el dataset Palmer Penguins
df = sns.load_dataset('penguins').dropna()

# Traducir los nombres de columna al español para que las consultas sean más naturales
df = df.rename(columns={
    'species':           'especie',
    'island':            'isla',
    'bill_length_mm':    'longitud_pico_mm',
    'bill_depth_mm':     'profundidad_pico_mm',
    'flipper_length_mm': 'longitud_aleta_mm',
    'body_mass_g':       'masa_corporal_g',
    'sex':               'sexo'
})

# Traducir los valores categóricos al español
df['sexo'] = df['sexo'].map({'male': 'macho', 'female': 'hembra'})

print(f"Dataset: {len(df)} filas, {len(df.columns)} columnas")
print(f"Columnas: {list(df.columns)}")
df.head()

Dataset: 333 filas, 7 columnas
Columnas: ['especie', 'isla', 'longitud_pico_mm', 'profundidad_pico_mm', 'longitud_aleta_mm', 'masa_corporal_g', 'sexo']


,especie,isla,longitud_pico_mm,profundidad_pico_mm,longitud_aleta_mm,masa_corporal_g,sexo
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,NaN
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,NaN
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,NaN
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,NaN


**Celda 4: Guardar el DataFrame en SQLite**

In [21]:
# Ruta del archivo de la base de datos
DB_PATH = 'pinguinos.db'

# Crear la conexión SQLite y guardar el DataFrame como tabla
conn = sqlite3.connect(DB_PATH)
df.to_sql('pinguinos', conn, if_exists='replace', index=False)
conn.close()

print(f"✅ Base de datos creada en: {DB_PATH}")
print(f"   Tamaño del archivo: {os.path.getsize(DB_PATH) / 1024:.1f} KB")

✅ Base de datos creada en: pinguinos.db
   Tamaño del archivo: 24.0 KB


**Explicación — `DataFrame.to_sql()`:**

`df.to_sql(name, con, if_exists, index)` escribe el DataFrame como una tabla en la base de datos:
- `name` — nombre de la tabla SQL
- `con` — conexión a la base de datos
- `if_exists='replace'` — si la tabla ya existe, la sobreescribe (alternativa: `'append'` para añadir filas)
- `index=False` — no escribe el índice de pandas como columna SQL

Es la forma más rápida de pasar de pandas a SQL. Funciona con SQLite, PostgreSQL, MySQL y cualquier base de datos compatible con SQLAlchemy.

**Celda 5: Explorar la Base de Datos con SQL puro**

In [22]:
# Verificar que la base de datos funciona correctamente con una consulta directa
conn = sqlite3.connect(DB_PATH)

# Ver el esquema de la tabla
esquema = pd.read_sql("PRAGMA table_info(pinguinos)", conn)
print("=== Esquema de la tabla 'pinguinos' ===")
print(esquema[['name', 'type']].to_string(index=False))

print("\n=== Conteo por especie ===")
resultado = pd.read_sql(
    "SELECT especie, COUNT(*) as total FROM pinguinos GROUP BY especie ORDER BY total DESC",
    conn
)
print(resultado.to_string(index=False))

conn.close()

=== Esquema de la tabla 'pinguinos' ===
               name type
            especie TEXT
               isla TEXT
   longitud_pico_mm REAL
profundidad_pico_mm REAL
  longitud_aleta_mm REAL
    masa_corporal_g REAL
               sexo TEXT

=== Conteo por especie ===
  especie  total
   Adelie    146
   Gentoo    119
Chinstrap     68


**Explicación:**

`PRAGMA table_info(nombre_tabla)` es una instrucción especial de SQLite que devuelve el esquema de una tabla: nombres de columnas, tipos de datos, si permiten nulos, etc. Usar `pd.read_sql(query, conn)` devuelve el resultado de cualquier consulta SELECT directamente como un DataFrame de pandas, lo que facilita el análisis posterior.

---
## Sección 3: Configurar Vanna

Vanna usa una arquitectura de **herencia múltiple de Python** para combinar de forma modular un vector store (para el contexto RAG) con un LLM (para la generación de SQL). Esto permite intercambiar cualquiera de los dos componentes sin cambiar el resto del código.

**Celda 6: Definir e Instanciar la Clase Vanna**

In [23]:
# Definir la clase combinando ChromaDB (vector store) + OpenAI (LLM)
class MiVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

# Instanciar con GPT-4o-mini (el modelo más barato y suficientemente potente para SQL)
vn = MiVanna(config={
    'api_key': OPENAI_API_KEY,
    'model':   'gpt-4o-mini'
})

# Conectar Vanna a la base de datos SQLite
vn.connect_to_sqlite(DB_PATH)

print("✅ Vanna configurado y conectado a la base de datos.")

✅ Vanna configurado y conectado a la base de datos.


**Explicación — Herencia múltiple en Vanna:**

La clase `MiVanna` hereda de dos padres:
- `ChromaDB_VectorStore` — gestiona la base de datos vectorial local donde Vanna guardará los embeddings del DDL, documentación y ejemplos SQL
- `OpenAI_Chat` — envía el prompt enriquecido a GPT-4o-mini y recibe la consulta SQL generada

Para cambiar a otro LLM solo tienes que cambiar la segunda clase padre. Por ejemplo, para usar Ollama localmente (sin coste):
```python
from vanna.ollama import Ollama
class MiVannaLocal(ChromaDB_VectorStore, Ollama):
    ...
vn = MiVannaLocal(config={'ollama_host': 'http://localhost:11434', 'model': 'llama3.1'})
```

`vn.connect_to_sqlite(ruta)` establece la conexión y permite a Vanna ejecutar el SQL generado directamente contra la base de datos.

---
## Sección 4: Entrenar a Vanna

"Entrenar" a Vanna no significa fine-tuning del LLM — el LLM ya está entrenado. Significa **poblar el vector store** con información sobre tu base de datos específica para que el RAG funcione correctamente. Cuanto mejor sea el contexto que le das, más precisas serán las consultas generadas.

Hay tres tipos de información que puedes añadir al vector store:
1. **DDL** — la definición de las tablas (CREATE TABLE)
2. **Documentación** — descripción en lenguaje natural de las tablas y columnas
3. **Ejemplos SQL** — pares (pregunta → SQL correcto) que sirven de referencia

**Celda 7: Entrenamiento con DDL (Esquema de la Tabla)**

In [24]:
# Entrenar con el DDL de la tabla: le dice a Vanna qué columnas existen y de qué tipo son
vn.train(ddl="""
    CREATE TABLE pinguinos (
        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'
        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'
        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros
        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros
        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros
        masa_corporal_g    REAL,    -- Masa corporal en gramos
        sexo               TEXT     -- Sexo: 'macho' o 'hembra'
    );
""")

print("✅ DDL añadido al vector store.")

Adding ddl: 
    CREATE TABLE pinguinos (
        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'
        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'
        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros
        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros
        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros
        masa_corporal_g    REAL,    -- Masa corporal en gramos
        sexo               TEXT     -- Sexo: 'macho' o 'hembra'
    );



Insert of existing embedding ID: d443d2e9-1b06-5b9a-aac1-c23fbb8aef94-ddl
Add of existing embedding ID: d443d2e9-1b06-5b9a-aac1-c23fbb8aef94-ddl


✅ DDL añadido al vector store.


**Explicación — DDL como contexto:**

El DDL (*Data Definition Language*) es el tipo de información más importante para Vanna. Le dice exactamente qué tablas y columnas existen, con sus tipos de datos. Los comentarios SQL (`--`) son especialmente valiosos porque el LLM los lee y entiende el significado de cada campo, lo que mejora la precisión de las consultas generadas.

Vanna convierte el DDL en un vector (embedding) usando el modelo de embeddings de OpenAI y lo almacena en ChromaDB. Cuando el usuario hace una pregunta, Vanna busca los DDL más similares semánticamente para incluirlos en el prompt.

**Celda 8: Entrenamiento con Documentación en Lenguaje Natural**

In [25]:
# Documentación general sobre el dataset
vn.train(documentation="""
La tabla 'pinguinos' contiene datos morfológicos de 333 pingüinos de tres especies
del Archipiélago Palmer, Antártida. Los datos fueron recopilados entre 2007 y 2009.

Valores posibles:
- especie: 'Adelie', 'Chinstrap', 'Gentoo'
- isla: 'Biscoe', 'Dream', 'Torgersen'
- sexo: 'macho', 'hembra'

Todas las medidas morfológicas (longitud_pico_mm, profundidad_pico_mm,
longitud_aleta_mm, masa_corporal_g) son valores numéricos continuos.
No hay valores nulos en ninguna columna.
""")

# Documentación específica sobre relaciones entre variables
vn.train(documentation="""
Relaciones conocidas en el dataset:
- Los pingüinos Gentoo son significativamente más grandes y pesados que Adelie y Chinstrap.
- La especie Chinstrap solo se encuentra en la isla Dream.
- La especie Gentoo solo se encuentra en la isla Biscoe.
- La especie Adelie se encuentra en las tres islas.
- Existe una correlación positiva fuerte entre longitud_aleta_mm y masa_corporal_g.
""")

print("✅ Documentación añadida al vector store.")

Adding documentation....


Add of existing embedding ID: 6a349257-9a48-5136-802c-58942a355b14-doc
Insert of existing embedding ID: 6a349257-9a48-5136-802c-58942a355b14-doc
Add of existing embedding ID: 9851623d-a91b-55aa-a7ab-42dd42281ea2-doc


Adding documentation....


Insert of existing embedding ID: 9851623d-a91b-55aa-a7ab-42dd42281ea2-doc


✅ Documentación añadida al vector store.


**Explicación — Documentación como contexto:**

La documentación complementa el DDL con información que no se puede expresar en SQL: valores posibles de columnas categóricas, relaciones entre variables, reglas de negocio, aclaraciones sobre el dominio. Esta información es crucial para que el LLM genere SQL correcto en situaciones ambiguas.

Por ejemplo, sin la documentación, si el usuario pregunta "¿qué especie es la más grande?", el LLM podría no saber si "grande" se refiere a `masa_corporal_g`, `longitud_aleta_mm` o alguna combinación. Con la documentación, el LLM tiene más contexto para tomar la decisión correcta.

**Celda 9: Entrenamiento con Ejemplos SQL**

In [26]:
# Los ejemplos SQL son los más potentes: muestran al LLM exactamente cómo
# se debe escribir SQL para este dataset específico

ejemplos_sql = [
    # Conteos y agrupaciones
    "SELECT especie, COUNT(*) AS total FROM pinguinos GROUP BY especie ORDER BY total DESC",
    "SELECT isla, especie, COUNT(*) AS total FROM pinguinos GROUP BY isla, especie ORDER BY isla",
    "SELECT sexo, COUNT(*) AS total FROM pinguinos GROUP BY sexo",

    # Estadísticas por grupo
    "SELECT especie, ROUND(AVG(masa_corporal_g), 1) AS masa_media FROM pinguinos GROUP BY especie ORDER BY masa_media DESC",
    "SELECT especie, ROUND(AVG(longitud_aleta_mm), 1) AS aleta_media, ROUND(AVG(longitud_pico_mm), 1) AS pico_medio FROM pinguinos GROUP BY especie",
    "SELECT isla, ROUND(AVG(masa_corporal_g), 1) AS masa_media, COUNT(*) AS total FROM pinguinos GROUP BY isla",

    # Comparaciones entre grupos
    "SELECT especie, sexo, ROUND(AVG(masa_corporal_g), 1) AS masa_media FROM pinguinos GROUP BY especie, sexo ORDER BY especie, sexo",
    "SELECT especie, MIN(masa_corporal_g) AS minimo, MAX(masa_corporal_g) AS maximo, ROUND(AVG(masa_corporal_g), 1) AS media FROM pinguinos GROUP BY especie",

    # Filtros con WHERE
    "SELECT * FROM pinguinos WHERE especie = 'Gentoo' AND sexo = 'macho' ORDER BY masa_corporal_g DESC LIMIT 10",
    "SELECT * FROM pinguinos WHERE masa_corporal_g > 5000 ORDER BY masa_corporal_g DESC",
    "SELECT COUNT(*) AS total FROM pinguinos WHERE isla = 'Biscoe' AND especie = 'Gentoo'",

    # Valores extremos
    "SELECT especie, isla, masa_corporal_g FROM pinguinos ORDER BY masa_corporal_g DESC LIMIT 5",
    "SELECT especie, longitud_pico_mm FROM pinguinos WHERE longitud_pico_mm = (SELECT MAX(longitud_pico_mm) FROM pinguinos)",
]

for sql in ejemplos_sql:
    vn.train(sql=sql)

print(f"✅ {len(ejemplos_sql)} ejemplos SQL añadidos al vector store.")

Using model gpt-4o-mini for 74.75 tokens (approx)


Question generated with sql: What is the total number of individuals for each species, sorted by the highest count? 
Adding SQL...
Using model gpt-4o-mini for 76.25 tokens (approx)
Question generated with sql: What is the total number of penguins by species on each island? 
Adding SQL...
Using model gpt-4o-mini for 68.25 tokens (approx)


Insert of existing embedding ID: 59277e5e-a088-537f-b5b3-990f14a8242e-sql
Add of existing embedding ID: 59277e5e-a088-537f-b5b3-990f14a8242e-sql


Question generated with sql: What is the total count of individuals by gender? 
Adding SQL...
Using model gpt-4o-mini for 82.75 tokens (approx)
Question generated with sql: What is the average body mass of each species, sorted from heaviest to lightest? 
Adding SQL...
Using model gpt-4o-mini for 89.0 tokens (approx)


Insert of existing embedding ID: e2e4d4a3-bb9a-5622-b4f0-f4c6957544e3-sql
Add of existing embedding ID: e2e4d4a3-bb9a-5622-b4f0-f4c6957544e3-sql


...

Question generated with sql: What are the top 10 heaviest male Gentoo penguins? 
Adding SQL...
Using model gpt-4o-mini for 74.0 tokens (approx)
Question generated with sql: What are the largest individuals with a body mass greater than 5000 grams? 
Adding SQL...
Using model gpt-4o-mini for 74.5 tokens (approx)


Insert of existing embedding ID: cc51bf0a-68fd-5970-8741-3e976b5d8aa3-sql
Add of existing embedding ID: cc51bf0a-68fd-5970-8741-3e976b5d8aa3-sql


Question generated with sql: How many Gentoo penguins are there on Biscoe Island? 
Adding SQL...
Using model gpt-4o-mini for 76.0 tokens (approx)
Question generated with sql: What are the top 5 penguin species with the highest body mass recorded on each island? 
Adding SQL...
Using model gpt-4o-mini for 83.0 tokens (approx)


Insert of existing embedding ID: e8f1e56e-db77-5091-a3c5-d3c2eff7aa0a-sql
Add of existing embedding ID: e8f1e56e-db77-5091-a3c5-d3c2eff7aa0a-sql


Question generated with sql: What species has the longest beak length? 
Adding SQL...
✅ 13 ejemplos SQL añadidos al vector store.


**Explicación — Ejemplos SQL como few-shot learning:**

Los ejemplos SQL funcionan como **few-shot examples** para el LLM: cuando el usuario hace una pregunta, Vanna recupera los ejemplos más similares semánticamente y los incluye en el prompt. El LLM "aprende" de ellos cómo debe escribir SQL para esta base de datos específica.

Los buenos ejemplos SQL deben cubrir:
- **Patrones de agrupación** (`GROUP BY`) — los más frecuentes en preguntas analíticas
- **Funciones de agregación** (`COUNT`, `AVG`, `MIN`, `MAX`)
- **Filtros** (`WHERE`) con los valores categóricos exactos que existen en los datos
- **Ordenación** (`ORDER BY`) y límites (`LIMIT`)

Cuanto más variados y representativos sean los ejemplos, mejor generalizará Vanna a preguntas nuevas.

**Celda 10: Verificar el Contenido del Vector Store**

In [27]:
# Consultar qué información tiene almacenada Vanna en el vector store
datos_entrenamiento = vn.get_training_data()

print(f"Total de elementos en el vector store: {len(datos_entrenamiento)}")
print("\nTipos de datos almacenados:")
print(datos_entrenamiento['training_data_type'].value_counts().to_string())
print("\nPrimeros registros:")
datos_entrenamiento.head(5)

Total de elementos en el vector store: 40

Tipos de datos almacenados:
training_data_type
sql              37
documentation     2
ddl               1

Primeros registros:


,id,question,content,training_data_type
0,62254b27-4c70-5b21-8649-f318b6559fce-sql,"What is the count of each species of penguin, ...","SELECT especie, COUNT(*) AS total FROM pinguin...",sql
1,d45bca94-371e-5091-81dc-20f78b6d466e-sql,What is the total number of each species of pe...,"SELECT isla, especie, COUNT(*) AS total FROM p...",sql
2,59277e5e-a088-537f-b5b3-990f14a8242e-sql,What is the total count of individuals by gender?,"SELECT sexo, COUNT(*) AS total FROM pinguinos ...",sql
3,d968b0f7-8235-5882-9731-3ab3792c53f5-sql,What is the average body weight of each pengui...,"SELECT especie, ROUND(AVG(masa_corporal_g), 1)...",sql
4,6f30802c-76d8-5776-8d49-aa32343242fb-sql,What is the average flipper length and average...,"SELECT especie, ROUND(AVG(longitud_aleta_mm), ...",sql


---
## Sección 5: Consultas en Lenguaje Natural

Con Vanna entrenado, podemos empezar a hacer preguntas en lenguaje natural. Vanna tiene varios métodos útiles:
- `vn.generate_sql(pregunta)` — solo genera el SQL, no lo ejecuta
- `vn.run_sql(sql)` — ejecuta el SQL y devuelve un DataFrame
- `vn.ask(pregunta)` — genera el SQL, lo ejecuta y opcionalmente genera un gráfico

**Celda 11: Primera Consulta — Generar y Ejecutar SQL**

In [28]:
# Pregunta 1: conteo básico por especie
pregunta = "¿Cuántos pingüinos hay de cada especie?"
print(f"Pregunta: {pregunta}")
print("-" * 50)

# Paso 1: generar el SQL
sql_generado = vn.generate_sql(pregunta)
print(f"SQL generado:\n{sql_generado}")
print("-" * 50)

# Paso 2: ejecutar el SQL y ver el resultado
resultado = vn.run_sql(sql_generado)
print("Resultado:")
resultado

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 2, updating n_results = 2


Pregunta: ¿Cuántos pingüinos hay de cada especie?
--------------------------------------------------
SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\n    CREATE TABLE pinguinos (\n        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'\n        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'\n        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros\n        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros\n        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros\n        masa_corporal_g    REAL,    -- Masa corporal en gramos\n        sexo               TEXT     -- Sexo: 'macho' o 'hembra'\n    );\n\n\n\n===Additional Context \n\n\nLa tabla 'pinguinos' co

,especie,total
0,Adelie,146
1,Gentoo,119
2,Chinstrap,68


**Celda 12: Consultas más Complejas**

In [29]:
# Una función auxiliar para mostrar pregunta → SQL → resultado de forma limpia
def consultar(pregunta):
    print(f"❓ Pregunta: {pregunta}")
    sql = vn.generate_sql(pregunta)
    print(f"🔍 SQL generado:\n   {sql.strip()}")
    resultado = vn.run_sql(sql)
    print(f"📊 Resultado ({len(resultado)} filas):")
    display(resultado)
    print()
    return resultado

# Consulta 2: estadísticas por grupo
consultar("¿Cuál es la masa corporal media de cada especie, ordenada de mayor a menor?")

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 2, updating n_results = 2


❓ Pregunta: ¿Cuál es la masa corporal media de cada especie, ordenada de mayor a menor?
SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\n    CREATE TABLE pinguinos (\n        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'\n        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'\n        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros\n        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros\n        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros\n        masa_corporal_g    REAL,    -- Masa corporal en gramos\n        sexo               TEXT     -- Sexo: 'macho' o 'hembra'\n    );\n\n\n\n===Additional Context \n\n\nLa tabla 'pinguinos' contiene datos 

,especie,masa_media
0,Gentoo,5092.4
1,Chinstrap,3733.1
2,Adelie,3706.2


,especie,masa_media
0,Gentoo,5092.4
1,Chinstrap,3733.1
2,Adelie,3706.2


In [30]:
# Consulta 3: filtro + estadísticas
consultar("¿Cuál es la longitud media de aleta y pico de los pingüinos macho de la isla Biscoe?")

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 2, updating n_results = 2


❓ Pregunta: ¿Cuál es la longitud media de aleta y pico de los pingüinos macho de la isla Biscoe?
SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\n    CREATE TABLE pinguinos (\n        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'\n        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'\n        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros\n        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros\n        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros\n        masa_corporal_g    REAL,    -- Masa corporal en gramos\n        sexo               TEXT     -- Sexo: 'macho' o 'hembra'\n    );\n\n\n\n===Additional Context \n\n\nRelaciones conocidas en el 

,aleta_media,pico_medio
0,None,None


,aleta_media,pico_medio
0,None,None


In [31]:
# Consulta 4: valores extremos
consultar("¿Cuáles son los 5 pingüinos más pesados y de qué especie son?")

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 2, updating n_results = 2


❓ Pregunta: ¿Cuáles son los 5 pingüinos más pesados y de qué especie son?
SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\n    CREATE TABLE pinguinos (\n        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'\n        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'\n        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros\n        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros\n        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros\n        masa_corporal_g    REAL,    -- Masa corporal en gramos\n        sexo               TEXT     -- Sexo: 'macho' o 'hembra'\n    );\n\n\n\n===Additional Context \n\n\nRelaciones conocidas en el dataset:\n- Los pingüin

,especie,masa_corporal_g
0,Gentoo,6300.0
1,Gentoo,6050.0
2,Gentoo,6000.0
3,Gentoo,6000.0
4,Gentoo,5950.0


,especie,masa_corporal_g
0,Gentoo,6300.0
1,Gentoo,6050.0
2,Gentoo,6000.0
3,Gentoo,6000.0
4,Gentoo,5950.0


In [32]:
# Consulta 5: pregunta de composición
consultar("¿Cuántos pingüinos machos y hembras hay en cada especie?")

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 2, updating n_results = 2


❓ Pregunta: ¿Cuántos pingüinos machos y hembras hay en cada especie?
SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\n    CREATE TABLE pinguinos (\n        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'\n        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'\n        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros\n        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros\n        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros\n        masa_corporal_g    REAL,    -- Masa corporal en gramos\n        sexo               TEXT     -- Sexo: 'macho' o 'hembra'\n    );\n\n\n\n===Additional Context \n\n\nLa tabla 'pinguinos' contiene datos morfológicos de 333

,especie,sexo,cantidad
0,Adelie,None,146
1,Chinstrap,None,68
2,Gentoo,None,119


,especie,sexo,cantidad
0,Adelie,None,146
1,Chinstrap,None,68
2,Gentoo,None,119


**Explicación — Cómo funciona el RAG en la práctica:**

Antes de llamar al LLM, Vanna hace lo siguiente:

1. Convierte la pregunta del usuario en un vector (embedding)
2. Busca en ChromaDB los N elementos más similares semánticamente (DDL, docs, ejemplos SQL)
3. Construye un prompt con ese contexto:
   ```
   Eres un experto en SQL. Usa el siguiente esquema y ejemplos para generar SQL:
   
   DDL: CREATE TABLE pinguinos (...)
   Documentación: La tabla pinguinos contiene...
   Ejemplo SQL: SELECT especie, COUNT(*)...
   
   Pregunta: ¿Cuántos pingüinos hay de cada especie?
   SQL:
   ```
4. El LLM completa el prompt generando solo la parte SQL

Si el SQL generado no te parece correcto, puedes mejorarlo añadiendo más ejemplos al vector store con `vn.train(sql="...")` o refinando la documentación.

**Celda 13: Visualizar Resultados con Plotly**

In [33]:
import plotly.express as px

# Consulta con resultado para visualizar
df_resultado = consultar("¿Cuál es la masa corporal media por especie y sexo?")

# Visualizar el resultado con Plotly
fig = px.bar(
    df_resultado,
    x='especie',
    y='masa_media',
    color='sexo',
    barmode='group',
    title="Masa corporal media por especie y sexo",
    labels={'masa_media': 'Masa media (g)', 'especie': 'Especie', 'sexo': 'Sexo'}
)
fig.show()

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 2, updating n_results = 2


❓ Pregunta: ¿Cuál es la masa corporal media por especie y sexo?
SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\n    CREATE TABLE pinguinos (\n        especie            TEXT,    -- Especie del pingüino: 'Adelie', 'Chinstrap' o 'Gentoo'\n        isla               TEXT,    -- Isla de origen: 'Biscoe', 'Dream' o 'Torgersen'\n        longitud_pico_mm   REAL,    -- Longitud del pico en milímetros\n        profundidad_pico_mm REAL,   -- Profundidad (grosor) del pico en milímetros\n        longitud_aleta_mm  REAL,    -- Longitud de la aleta en milímetros\n        masa_corporal_g    REAL,    -- Masa corporal en gramos\n        sexo               TEXT     -- Sexo: 'macho' o 'hembra'\n    );\n\n\n\n===Additional Context \n\n\nLa tabla 'pinguinos' contiene datos morfológicos de 333 ping

,especie,sexo,masa_media
0,Adelie,None,3706.2
1,Chinstrap,None,3733.1
2,Gentoo,None,5092.4


**Explicación:**

El resultado de `vn.run_sql()` es siempre un DataFrame de pandas estándar, lo que significa que puedes usar directamente toda la maquinaria de visualización que ya conoces: Plotly, Matplotlib, Seaborn. Esta es una ventaja clave de Vanna: no impone un formato de salida propietario.

En la sección siguiente, integraremos esta cadena completa (pregunta → SQL → DataFrame → gráfico) en una interfaz Streamlit conversacional.

---
# Parte 2: Chatbot Interactivo con Streamlit

Las siguientes celdas construyen `chatbot_sql.py` usando `%%writefile`.
Streamlit no puede ejecutarse dentro de Jupyter — se lanza como servidor independiente desde la terminal.

**Antes de lanzar la app, asegúrate de:**
1. Haber ejecutado todas las celdas `%%writefile` de esta sección (generan el archivo `chatbot_sql.py`).
2. Que el archivo `pinguinos.db` esté en la misma carpeta (se crea en la Sección 2).

Luego, abre una terminal en la carpeta del notebook y ejecuta:

```bash
streamlit run chatbot_sql.py
```

Streamlit abrirá automáticamente el navegador en `http://localhost:8501`.


---
## Sección 6: Chatbot de SQL con Streamlit

Vamos a construir una aplicación Streamlit que permita al usuario hacer preguntas en lenguaje natural a la base de datos a través de una interfaz de chat. Esta sección conecta directamente con el laboratorio de Streamlit anterior:
- Usaremos `st.chat_message` y `st.chat_input` — los componentes de chat de Streamlit
- Usaremos `st.session_state` — el bonus del laboratorio anterior — para mantener el historial de la conversación entre reruns

Como en el laboratorio anterior, usaremos `%%writefile` para construir la app.

**Celda 14: Crear la App Streamlit — Estructura Base**

In [34]:
%%writefile chatbot_sql.py
# ============================================================
# Chatbot de SQL con Vanna + Streamlit
# Laboratorio Text-to-SQL - Máster en IA / Deep Learning
# ============================================================

import streamlit as st
import pandas as pd
import plotly.express as px

from vanna.openai import OpenAI_Chat
from vanna.chromadb import ChromaDB_VectorStore

# --- Configuración de la página ---
st.set_page_config(
    page_title="Asistente SQL de Pingüinos",
    page_icon="🐧",
    layout="wide"
)

# --- Clase Vanna (igual que en el notebook) ---
class MiVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

Writing chatbot_sql.py


**Celda 15: Inicializar Vanna con Caché**

In [35]:
%%writefile -a chatbot_sql.py

# --- API Key en la barra lateral (sin variables de entorno) ---
api_key_input = st.sidebar.text_input(
    "🔑 OpenAI API Key",
    type="password",
    placeholder="sk-..."
)
if not api_key_input:
    st.info("👈 Introduce tu OpenAI API Key en el panel lateral para comenzar.")
    st.stop()

# @st.cache_resource: para objetos pesados que no se pueden serializar
# Recibe la api_key como argumento para que el caché se invalide si cambia
@st.cache_resource
def inicializar_vanna(api_key):
    vn = MiVanna(config={'api_key': api_key, 'model': 'gpt-4o-mini'})
    vn.connect_to_sqlite('pinguinos.db')
    return vn

vn = inicializar_vanna(api_key_input)


Appending to chatbot_sql.py


**Explicación — API Key con `st.sidebar.text_input`:**

En lugar de variables de entorno, el alumno pega su API key directamente en la barra lateral. `type="password"` oculta los caracteres para que no quede visible en pantalla.

**`@st.cache_resource`** cachea la instancia de Vanna. Pasamos `api_key` como argumento para que, si el usuario cambia la clave, el caché se invalide y se cree una instancia nueva con la clave correcta.

**`@st.cache_resource` vs `@st.cache_data`:**
- `@st.cache_data` — para DataFrames y datos serializables (hace una copia)
- `@st.cache_resource` — para modelos, conexiones y clientes de API (devuelve el objeto directamente)

**Celda 16: Encabezado y Sidebar Informativo**

In [36]:
%%writefile -a chatbot_sql.py

# --- Encabezado ---
st.title("🐧 Asistente SQL de Pingüinos")
st.markdown(
    "Pregunta en español sobre el dataset de pingüinos y obtén resultados directamente "
    "de la base de datos. La IA traducirá tu pregunta a SQL automáticamente."
)

# --- Sidebar con información y preguntas de ejemplo ---
st.sidebar.header("💡 Preguntas de ejemplo")
preguntas_ejemplo = [
    "¿Cuántos pingüinos hay de cada especie?",
    "¿Cuál es la masa media por especie?",
    "¿Qué especie tiene el pico más largo en promedio?",
    "¿Cuántos pingüinos macho y hembra hay en cada isla?",
    "Muestra los 5 pingüinos más pesados con su especie y isla.",
    "¿Cuál es la longitud media de aleta de los Gentoo machos?",
]
for p in preguntas_ejemplo:
    st.sidebar.markdown(f"• *{p}*")

st.sidebar.divider()
st.sidebar.markdown("**Base de datos:** `pinguinos.db` (SQLite)")
st.sidebar.markdown("**Tabla:** `pinguinos` (333 registros)")
st.sidebar.markdown("**Modelo:** GPT-4o-mini + ChromaDB RAG")

Appending to chatbot_sql.py


**Celda 17: Historial del Chat con Session State**

In [37]:
%%writefile -a chatbot_sql.py

# --- Historial del chat (session_state) ---
# Cada mensaje es un dict: {'role': 'user'/'assistant', 'content': texto, 'df': DataFrame|None, 'sql': str|None}
if 'mensajes' not in st.session_state:
    st.session_state.mensajes = [
        {
            'role': 'assistant',
            'content': '¡Hola! Soy tu asistente SQL. Hazme cualquier pregunta sobre los pingüinos del Archipiélago Palmer y consultaré la base de datos automáticamente.',
            'df': None,
            'sql': None
        }
    ]

# Renderizar el historial de mensajes anteriores
for mensaje in st.session_state.mensajes:
    with st.chat_message(mensaje['role']):
        st.markdown(mensaje['content'])
        # Si el mensaje tiene SQL generado, mostrarlo en un expander
        if mensaje['sql']:
            with st.expander("🔍 Ver SQL generado"):
                st.code(mensaje['sql'], language='sql')
        # Si el mensaje tiene un DataFrame resultado, mostrarlo
        if mensaje['df'] is not None and not mensaje['df'].empty:
            st.dataframe(mensaje['df'], use_container_width=True)
            # Si el DataFrame tiene exactamente 2 columnas y la segunda es numérica, graficar
            cols = mensaje['df'].columns.tolist()
            if len(cols) == 2 and pd.api.types.is_numeric_dtype(mensaje['df'][cols[1]]):
                fig = px.bar(mensaje['df'], x=cols[0], y=cols[1])
                st.plotly_chart(fig, use_container_width=True)

Appending to chatbot_sql.py


**Explicación — `st.chat_message` y el historial:**

**`st.chat_message(role)`** crea un bloque visual de chat con el icono y estilo del rol especificado:
- `'user'` — burbuja del usuario (icono de persona)
- `'assistant'` — burbuja del asistente (icono de robot)

El historial de mensajes se almacena en `st.session_state.mensajes`. Cada rerun de Streamlit re-renderiza todos los mensajes del historial, lo que crea la ilusión de una conversación persistente. Sin `session_state`, el historial se resetearía con cada pregunta y solo veríamos el último mensaje.

Cada mensaje guarda no solo el texto sino también el SQL generado y el DataFrame resultado, para poder re-renderizarlos correctamente al volver a cargar el historial.

**Celda 18: Procesar la Pregunta del Usuario**

In [38]:
%%writefile -a chatbot_sql.py

# --- Input del usuario ---
pregunta = st.chat_input("Escribe tu pregunta sobre los pingüinos...")

if pregunta:
    # 1. Añadir el mensaje del usuario al historial y mostrarlo
    st.session_state.mensajes.append({'role': 'user', 'content': pregunta, 'df': None, 'sql': None})
    with st.chat_message('user'):
        st.markdown(pregunta)

    # 2. Procesar la pregunta con Vanna
    with st.chat_message('assistant'):
        with st.spinner('Generando SQL y consultando la base de datos...'):
            try:
                # Generar SQL a partir de la pregunta
                sql_generado = vn.generate_sql(pregunta)

                # Ejecutar el SQL
                df_resultado = vn.run_sql(sql_generado)

                # Construir la respuesta de texto
                if df_resultado is not None and not df_resultado.empty:
                    respuesta = f"He encontrado **{len(df_resultado)} resultado(s)**:"
                else:
                    respuesta = "La consulta no devolvió resultados. Intenta reformular la pregunta."
                    df_resultado = None

            except Exception as e:
                respuesta = f"❌ No he podido procesar esa pregunta. Error: {str(e)}\n\nIntenta reformularla o usa una de las preguntas de ejemplo del panel lateral."
                sql_generado = None
                df_resultado = None

        # 3. Mostrar la respuesta
        st.markdown(respuesta)

        if sql_generado:
            with st.expander("🔍 Ver SQL generado"):
                st.code(sql_generado, language='sql')

        if df_resultado is not None and not df_resultado.empty:
            st.dataframe(df_resultado, use_container_width=True)
            # Gráfico automático para resultados simples de 2 columnas
            cols = df_resultado.columns.tolist()
            if len(cols) == 2 and pd.api.types.is_numeric_dtype(df_resultado[cols[1]]):
                fig = px.bar(
                    df_resultado, x=cols[0], y=cols[1],
                    title=pregunta,
                    labels={cols[0]: cols[0].replace('_', ' ').title(),
                            cols[1]: cols[1].replace('_', ' ').title()}
                )
                st.plotly_chart(fig, use_container_width=True)

        # 4. Guardar la respuesta del asistente en el historial
        st.session_state.mensajes.append({
            'role': 'assistant',
            'content': respuesta,
            'sql': sql_generado,
            'df': df_resultado
        })

Appending to chatbot_sql.py


**Explicación — Flujo completo del chatbot:**

**`st.chat_input(placeholder)`** renderiza un cuadro de texto fijo en la parte inferior de la pantalla (como los chats modernos). Cuando el usuario envía una pregunta, devuelve el texto y provoca un rerun del script.

**`st.spinner(texto)`** muestra un indicador de carga mientras se ejecuta el bloque `with`. Es importante usarlo durante las llamadas a la API de OpenAI (que pueden tardar 2-5 segundos) para que el usuario sepa que la app está procesando.

El bloque `try/except` es fundamental en una app de producción: si Vanna genera SQL inválido o la pregunta es demasiado ambigua, capturamos el error y mostramos un mensaje amigable en lugar de dejar que la app crashee.

El **gráfico automático** detecta si el resultado tiene exactamente dos columnas donde la segunda es numérica — el patrón más común en consultas de tipo "¿cuánto/cuál es el promedio de X por Y?". En ese caso, genera automáticamente un gráfico de barras sin que el usuario lo pida.

**Celda 19: Botón para Limpiar el Historial**

In [39]:
%%writefile -a chatbot_sql.py

# --- Botón para limpiar el historial del chat ---
st.sidebar.divider()
if st.sidebar.button("🗑️ Limpiar conversación"):
    st.session_state.mensajes = [
        {
            'role': 'assistant',
            'content': '¡Conversación reiniciada! ¿En qué puedo ayudarte?',
            'df': None,
            'sql': None
        }
    ]
    st.rerun()   # forzar un rerun para que se actualice el chat inmediatamente

Appending to chatbot_sql.py


**Explicación — `st.rerun()`:**

`st.rerun()` fuerza Streamlit a re-ejecutar el script inmediatamente desde el principio. Es útil después de modificar `session_state` desde un botón cuando quieres que el cambio se refleje en la UI en el mismo ciclo, sin esperar a la próxima interacción del usuario.

### 🔁 Checkpoint — Ejecutar el Chatbot

Abre una terminal en la carpeta donde está `chatbot_sql.py` y ejecuta:

```bash
streamlit run chatbot_sql.py
```

**Prueba estas interacciones:**
1. Escribe una de las preguntas de ejemplo del panel lateral.
2. Haz clic en el expander "Ver SQL generado" para ver el SQL producido por Vanna.
3. Prueba una pregunta más compleja que no esté en los ejemplos.
4. Haz varias preguntas seguidas y observa que el historial persiste.
5. Limpia la conversación con el botón del sidebar.

---

## Sección 7: Gestión del Vector Store y Casos Límite

Un sistema Text-to-SQL en producción necesita mantenimiento: añadir ejemplos cuando falla, eliminar ejemplos incorrectos y gestionar el ciclo de vida del vector store.

**Celda 20: Añadir Ejemplos cuando Vanna Falla**

In [40]:
# Si Vanna genera SQL incorrecto para una pregunta específica,
# añade el SQL correcto como ejemplo y Vanna lo usará en el futuro

# Ejemplo: si falla la pregunta sobre correlación entre variables
vn.train(sql="""
    SELECT
        especie,
        ROUND(AVG(longitud_aleta_mm), 1) AS aleta_media,
        ROUND(AVG(masa_corporal_g), 1)   AS masa_media,
        COUNT(*) AS total
    FROM pinguinos
    GROUP BY especie
    ORDER BY masa_media DESC
""")

# Verificar que el ejemplo se ha añadido
datos = vn.get_training_data()
print(f"Total de elementos en el vector store: {len(datos)}")

Using model gpt-4o-mini for 112.75 tokens (approx)
Question generated with sql: What is the average fin length and body mass for each species, along with the total count of individuals? 
Adding SQL...
Total de elementos en el vector store: 41


**Celda 21: Eliminar Ejemplos Incorrectos**

In [41]:
# Ver todos los elementos del vector store con sus IDs
datos = vn.get_training_data()
print("Elementos en el vector store:")
datos[['id', 'training_data_type', 'content']].head(10)

Elementos en el vector store:


,id,training_data_type,content
0,62254b27-4c70-5b21-8649-f318b6559fce-sql,sql,"SELECT especie, COUNT(*) AS total FROM pinguin..."
1,d45bca94-371e-5091-81dc-20f78b6d466e-sql,sql,"SELECT isla, especie, COUNT(*) AS total FROM p..."
2,59277e5e-a088-537f-b5b3-990f14a8242e-sql,sql,"SELECT sexo, COUNT(*) AS total FROM pinguinos ..."
3,d968b0f7-8235-5882-9731-3ab3792c53f5-sql,sql,"SELECT especie, ROUND(AVG(masa_corporal_g), 1)..."
4,6f30802c-76d8-5776-8d49-aa32343242fb-sql,sql,"SELECT especie, ROUND(AVG(longitud_aleta_mm), ..."
5,23f05834-3409-5e0f-972a-6bfd6f38b66a-sql,sql,"SELECT isla, ROUND(AVG(masa_corporal_g), 1) AS..."
6,4330b34c-33bb-5188-b5f2-1baa24f84ac7-sql,sql,"SELECT especie, sexo, ROUND(AVG(masa_corporal_..."
7,c55c1e99-4d57-5307-8711-2ff8ca215bdb-sql,sql,"SELECT especie, MIN(masa_corporal_g) AS minimo..."
8,120ba631-5587-5e8f-b840-27fab568fe7d-sql,sql,SELECT * FROM pinguinos WHERE especie = 'Gento...
9,ae4dc2ee-d294-5790-8bb3-dfdeb8a5b0d4-sql,sql,SELECT * FROM pinguinos WHERE masa_corporal_g ...


In [42]:
# Si quieres eliminar un elemento incorrecto, usa su ID:
# vn.remove_training_data(id='ID_DEL_ELEMENTO')

# Para eliminar TODOS los datos de entrenamiento y empezar desde cero:
# for _, fila in vn.get_training_data().iterrows():
#     vn.remove_training_data(id=fila['id'])

print("Para eliminar elementos, descomenta las líneas de arriba con el ID correspondiente.")

Para eliminar elementos, descomenta las líneas de arriba con el ID correspondiente.


**Explicación — Ciclo de mejora continua:**

Un sistema Text-to-SQL se mejora iterativamente:
1. **Prueba** con preguntas reales de los usuarios
2. **Identifica** los casos donde el SQL generado es incorrecto
3. **Escribe** el SQL correcto para esos casos
4. **Añade** el SQL correcto al vector store con `vn.train(sql=...)`
5. **Vuelve** al paso 1

Este proceso es mucho más rápido y barato que fine-tuning del LLM y produce mejoras inmediatas. En producción, puedes implementar un mecanismo de feedback donde los usuarios marquen si el resultado fue correcto, y los casos incorrectos se añaden automáticamente a una cola de revisión.

## Ejercicio Propuesto: Amplía el Sistema

Intenta implementar al menos **dos** de las siguientes mejoras:

### Nivel básico
1. **Segunda tabla:** Crea una tabla `islas` con información adicional sobre cada isla (coordenadas, temperatura media, etc.) y entrena a Vanna con el DDL del JOIN entre ambas tablas.

2. **Exportar resultados:** Añade al chatbot un `st.download_button` que aparezca después de cada respuesta con datos para descargar el DataFrame como CSV.

3. **Más ejemplos SQL:** Añade 10 ejemplos SQL más al vector store, especialmente para preguntas que impliquen subconsultas o cálculos de percentiles.

### Nivel intermedio
4. **Feedback de usuario:** Añade botones 👍/👎 debajo de cada respuesta del asistente. Si el usuario da 👎, guarda la pregunta en un archivo de log para revisión manual.

5. **Historial persistente:** Guarda el historial del chat en un archivo JSON para que persista entre sesiones (no solo en `session_state`, que se reinicia al cerrar el navegador).

### Nivel avanzado
6. **Vanna con Ollama:** Cambia el backend de OpenAI a Ollama con un modelo local (Llama 3.1 o similar). Compara la calidad del SQL generado con GPT-4o-mini.
   ```python
   from vanna.ollama import Ollama
   class MiVannaLocal(ChromaDB_VectorStore, Ollama):
       ...
   vn = MiVannaLocal(config={'ollama_host': 'http://localhost:11434', 'model': 'llama3.1'})
   ```

7. **Dataset propio:** Carga un CSV de tu elección en SQLite y entrena a Vanna con él. Construye un chatbot personalizado para ese dominio.

---
## Conclusiones

En este laboratorio has construido un sistema Text-to-SQL completo con Vanna.ai:

### Conceptos aprendidos

| Concepto | Descripción |
|---------|-------------|
| **Text-to-SQL** | Traducción automática de lenguaje natural a SQL usando LLMs |
| **RAG** | Retrieval-Augmented Generation: recuperar contexto relevante antes de generar |
| **Vector Store** | Base de datos vectorial (ChromaDB) para búsqueda semántica de contexto |
| **SQLite** | Base de datos embebida, perfecta para prototipos locales |
| **`DataFrame.to_sql()`** | Pasar datos de pandas a una base de datos SQL en una línea |
| **Entrenamiento Vanna** | DDL + Documentación + Ejemplos SQL = contexto RAG |

### Componentes nuevos de Streamlit

| Componente | Uso |
|-----------|-----|
| `@st.cache_resource` | Caché para objetos pesados no serializables (modelos, conexiones) |
| `st.chat_message(role)` | Burbujas de chat para usuario y asistente |
| `st.chat_input(placeholder)` | Cuadro de texto fijo en la parte inferior para el chat |
| `st.spinner(texto)` | Indicador de carga durante operaciones lentas |
| `st.rerun()` | Forzar un rerun inmediato del script |

### Conexión entre laboratorios

```
Lab Streamlit            Lab Text-to-SQL
─────────────            ──────────────
@st.cache_data     →     @st.cache_resource (para objetos más complejos)
st.session_state   →     Historial del chat (caso de uso real)
st.plotly_chart    →     Gráficos automáticos del resultado SQL
%%writefile        →     Mismo patrón para construir la app
```

### Próximos pasos

- **Vanna con tu propia base de datos:** Conecta Vanna a PostgreSQL o MySQL con `vn.connect_to_postgres(...)` o `vn.connect_to_mysql(...)`
- **Despliegue:** Publica el chatbot en Streamlit Community Cloud (gratuito) o en un servidor con Docker
- **Seguridad:** En producción, valida siempre el SQL generado antes de ejecutarlo para evitar SQL injection o consultas destructivas

---
### 🎉 ¡Felicidades! Has construido un chatbot de SQL con Inteligencia Artificial.